In [2278]:
import pandas as pd
import json

In [2279]:
## !pip install jinja2
##!pip install the matplotlib
##!pip install pandas openpyxl

In [2280]:
from datetime import datetime
import numpy as np

Cargar el fichero  excel

In [2281]:
df = pd.read_excel('E:/osley/Barecelona activa/curso especializacion analisis de datos/sprint 8/sprint10_complex.xlsx', header=3)

# Nivel 1 - EDA estructurado y limpieza básica

## Ejercicio 1 Exploración inicial del dataset


In [2282]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  1007 non-null   int64  
 1   Nom                         1006 non-null   str    
 2   Cognoms                     1006 non-null   str    
 3   DNI                         1006 non-null   str    
 4   País d'origen               1006 non-null   str    
 5   Ciutat                      1006 non-null   str    
 6   Dia de Naixement            1006 non-null   float64
 7   Mes de Naixement            1006 non-null   float64
 8   Any de Naixement            1006 non-null   float64
 9   Gènere                      966 non-null    str    
 10  Salari mensual              1006 non-null   str    
 11  Fills                       419 non-null    object 
 12  No Fills                    585 non-null    object 
 13  Grup Professional           1005 non-null   

El dataset tiene 1007 registros y 20 columnas, tipos de datos float64(7), int64(1), object(3), str(9)

In [2283]:
display(df.head(10))

,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1
5,5,Alexia,Puig,25014631B,Espanya,Las Palmas de Gran Canaria,1.0,8.0,1978.0,D,"2,069 â‚¬",NaN,True,Grup B,Cap de projecte,0.0,NaN,NaN,NaN,19.9
6,6,David,GÃ³mez Vila,36998038E,Espanya,Palma,25.0,9.0,1971.0,D,"2,349 â‚¬",NaN,True,Grup B,Cap de projecte,NaN,True,6196.0,10.6,17.8
7,7,Carmen,LÃ³pez,47376585K,Espanya,Barcelona,8.0,9.0,1962.0,H,"2,855 â‚¬",True,NaN,Grup C,Responsable de vendes,4.0,True,20000.0,5.9,15.4
8,540,Nora,Andersen Pedersen,37289414J,Noruega,Stavanger,8.0,3.0,1963.0,D,"1,338 â‚¬",NaN,True,Grup B,Analista junior,0.0,False,NaN,NaN,7.6
9,9,Alba,GarcÃ­a,75181648S,Espanya,Las Palmas de Gran Canaria,32.0,0.0,1940.0,H,"1,450 â‚¬",True,NaN,Grup A,Senior analyst,4.0,False,NaN,NaN,20.1


Hay filas vacias, filas con datos inconguentes, errores tipograficos, celdas sin datos, generos invertidos, fechas incorrectas, registros duplicados, tipos de datos que hay que cambiar
Hay una Columna que tine la misma numeracion que el indice, varias colunnas que se pueden representar en un una sola columna(columnas de tiempo y de hijos). 

## Ejercicio 2 Validaciones básicas de los datos y decisiones de eliminación

Eliminacion de columna 'Unnamed' que tiene los mis valores que el indice

In [2284]:
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

Eliminar la fila 1006 por tener todos sus valores Nan

In [2285]:
df.dropna(how='all', inplace=True)
df = df.drop(1006)

Arreglar el nombre de la columna CÃ\xa0rrec' por 'Carrec'

In [2286]:
df.rename(columns={'CÃ\xa0rrec': 'Carrec'}, inplace=True)
df.dropna(how='all', inplace=True)

Definimos dicionario con los errores tipograficos y su corección

In [2287]:
mapeo_caracteres = {
    'Ã\xa0': 'à', 'Ã©': 'é', 'Ã\xad': 'í', 'Ãº': 'ú', 'Ã‰': 'Á', 
    'Ã±': 'ñ', 'Ã³': 'ó', 'Ã¡': 'á', 'Ã¨': 'e', 'Ã\x81': 'A', 
    'Ã¤': 'ä', 'Ã¼': 'u', 'Ã¶': 'ö', 'Ã§': 'ci', 'Ã¸': 'ø'
}


Función de limpieza de texto optimizada con diccionario

Asegurar tipo string y reemplazar todo el diccionario en un solo paso

In [2288]:


def corregir_subcadenas_campo(dataframe, columnas_a_corregir, diccionario_cambios):
    for col in columnas_a_corregir:
        if col in dataframe.columns:
            
            dataframe[col] = (dataframe[col]
                              .astype(str)
                              .str.replace(diccionario_cambios, regex=True))


Llammos la funcion y con el dataframe, la lista de campos y el dicionarios con las correciones

In [2289]:
lista_campos = ['Nom', 'Cognoms', 'Ciutat', 'Carrec', "País d'origen"]
corregir_subcadenas_campo(df, lista_campos, mapeo_caracteres)

In [2290]:
display(df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Carrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7
3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,Sara,Serra,90585882A,Espanya,Tarragona,28.0,5.0,2002.0,D,"1,073 â‚¬",NaN,True,Grup A,Administratiu,0.0,False,NaN,NaN,15.8
1001,Kasper,Pettersen,62042563B,Noruega,Oslo,11.0,5.0,1996.0,D,2.290 â‚¬,NaN,True,Grup B,Senior analyst,0.0,True,80000.0,0.5,6.5
1002,Alba,Muñoz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,Claudia,Gil Ferrer,55297384F,Espanya,Valencia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3


Corección de Genero

Homogenizando la nomenclatura para los generos

In [2291]:

df['Gènere'] = df['Gènere'].replace({'H': 'Home', 'D': 'Dona', 'F': 'Dona'})


agrupamos todos los nombres que tienen genero incorecto, haciendo dos conjuntos un para las Mujeres y otro para los Hombres

Dicionario de Generos

In [2292]:
diccionario_generos = {
    # Nombres de mujer ('Dona')
    'Adri': 'Dona', 'Aina': 'Dona', 'Alba': 'Dona', 'Alexia': 'Dona', 'Anna': 'Dona', 
    'Camille': 'Dona', 'Carmen': 'Dona', 'Charlotte': 'Dona', 'Chiara': 'Dona', 
    'Clara': 'Dona', 'Claudia': 'Dona', 'Elena': 'Dona', 'Élise': 'Dona', 
    'Francesca': 'Dona', 'Gabriele': 'Dona', 'Giulia': 'Dona', 'Hannah': 'Dona', 
    'Ilaria': 'Dona', 'Irene': 'Dona', 'Joana': 'Dona', 'Katharina': 'Dona', 
    'Laia': 'Dona', 'Léa': 'Dona', 'Lucía': 'Dona', 'Marie': 'Dona', 'Marina': 'Dona', 
    'Marta': 'Dona', 'Martina': 'Dona', 'Mia': 'Dona', 'Núria': 'Dona', 'Sara': 'Dona', 
    'Simone': 'Dona', 'Sofie': 'Dona', 'Valentina': 'Dona',

    # Nombres de hombre ('Home')
    'Adrià': 'Home', 'Alejandro': 'Home', 'Alessandro': 'Home', 'Antoine': 'Home', 
    'Ben': 'Home', 'Carlos': 'Home', 'David': 'Home', 'Elias': 'Home', 'Erik': 'Home', 
    'Felix': 'Home', 'Guillem': 'Home', 'Hugo': 'Home', 'Jordi': 'Home', 'Lars': 'Home', 
    'Luca': 'Home', 'Lucas': 'Home', 'Magnus': 'Home', 'Marc': 'Home', 'Marco': 'Home', 
    'Matteo': 'Home', 'Max': 'Home', 'Nicolas': 'Home', 'Nil': 'Home', 'Noa': 'Home', 
    'Noah': 'Home', 'Oriol': 'Home', 'Paolo': 'Home', 'Pau': 'Home', 'Paul': 'Home', 
    'Pol': 'Home', 'Riccardo': 'Home', 'Sergio': 'Home', 'Thomas': 'Home', 
    'Víctor': 'Home', 'Xavier': 'Home'
}


Aplicamos el diccionario mediante map al dataframe

In [2293]:
df['Gènere'] = df['Nom'].map(diccionario_generos).combine_first(df['Gènere'])

In [2294]:
df['DNI'] = df['DNI'].str.strip()
df['DNI'] = df['DNI'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

Identificacion y Eliminacion de registros duplicados

In [2295]:
df_todos_duplicados = df[df.duplicated(subset=['DNI'], keep=False)]
df_duplicados_agrupados = df_todos_duplicados.sort_values(by='DNI')

print(f"Se encontraron {len(df_duplicados_agrupados)} filas asociadas a DNIs duplicados:\n")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    display(df_duplicados_agrupados)


Se encontraron 24 filas asociadas a DNIs duplicados:



,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Carrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
604,Laia,Navarro,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,Dona,"1,676 â‚¬",True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2
250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,Dona,1235 euros,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3
759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.0,1.0,1968.0,Home,"2,000 â‚¬",NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7
758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.0,1.0,1968.0,Home,1879 euros,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN
754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 â‚¬,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
654,Oriol,Gómez,16618423V,Espanya,Madrid,10.0,1.0,1968.0,Home,1731 euros,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN
692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,1663â‚¬,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9
558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,"1,176 â‚¬",NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0
757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,Home,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,Home,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9


In [2296]:
df_resultado = df[df['DNI'] == '15909601K']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

      Nom       Cognoms        DNI País d'origen  Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional           Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
250  Laia  Navarro Vila  15909601K       Espanya  Bilbao              30.0               5.0            1958.0   Dona     1235 euros  True     True            Grup D  Analista junior           0.0    False        NaN                   NaN                        10.3
604  Laia       Navarro  15909601K       Espanya  Bilbao              30.0               5.0            1958.0   Dona      1,676 â‚¬  True    False            Grup A         Analista           1.0     True       50.0                   NaN                        14.2


Con e DNI: 15909601K , 
- eliminar el registro 250 porque en es la misma persona que ha tenido hijo y ha mejorado en el cargo, ademas adquirio coche

In [2297]:
df = df.drop(250)

In [2298]:
df_resultado = df[df['DNI'] == '16618423V']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

       Nom Cognoms        DNI País d'origen     Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
654  Oriol   Gómez  16618423V       Espanya     Madrid              10.0               1.0            1968.0   Home     1731 euros  True      NaN            Grup B           Data analyst           4.0      NaN        NaN                   NaN                         NaN
754  Oriol   Gómez  16618423V       Espanya  Barcelona              10.0               1.0            1968.0   Home      2.358 â‚¬  True      NaN               NaN                    NaN           4.0      NaN        NaN                   NaN                         NaN
758  Oriol   Gómez  16618423V       Espanya     Màlaga              10.0               1.0            1968.0   Home     1879 euros  True      NaN            Grup B           Data Analyst 

Con e DNI: 16618423V , eliminar el registro 
- Eliminamos los registros 654,754,,758, al ser la misma persona que ha cambiado de ciudad, ha tenido un hijo mas y cambiado de trabajo, por loq ue queda el registro 759 

In [2299]:
df = df.drop([654,754,758])

In [2300]:
df_resultado = df[df['DNI'] == '49529438G']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

        Nom     Cognoms        DNI País d'origen  Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional           Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
558  Alexia  Pérez Díaz  49529438G       Espanya  Màlaga              12.0              10.0            1981.0   Dona      1,176 â‚¬   NaN     True            Grup A     Data Analyst           0.0    False        NaN                   NaN                        38.0
692  Alexia  Pérez Díaz  49529438G       Espanya  Màlaga              12.0              10.0            1981.0   Dona        1663â‚¬   NaN     True            Grup C  Cap de projecte           NaN     True    80000.0                  10.7                        16.9


Con e DNI: 49529438G , 
- eliminar el registro 558 porque es la mims persona que ha cambiado de cargo y ha comprado un coche

In [2301]:
df = df.drop(558)

In [2302]:
df_resultado = df[df['DNI'] == '52590001R']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

        Nom Cognoms        DNI País d'origen    Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional    Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
757   Oriol  Ferrer  52590001R       Espanya  Zaragoza              15.0               7.0            1973.0   Home         786â‚¬   NaN     True            Grup A  Analista           0.0     True    81381.0                   6.5                        13.9
1004  Oriol  Ferrer  52590001R       Espanya  Zaragoza              15.0               7.0            1973.0   Home         786â‚¬   NaN     True            Grup A  Analista           0.0     True    81381.0                   6.5                        13.9


Con e DNI: 52590001R
- eliminar el registro 1004 pues los dos registros son identicos por lo que no parta nada nuevo

In [2303]:
df = df.drop(1004)

In [2304]:
df_resultado = df[df['DNI'] == '54013571W']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

         Nom      Cognoms        DNI País d'origen    Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional        Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
89   Guillem  Pujol Costa  54013571W       Espanya  A Coruña              17.0               7.0            1983.0   Home      1.574 â‚¬  True      NaN            Grup B  Project lead           4.0    False        NaN                   NaN                        14.6
142    Marta         Ruiz  54013571W       Espanya     Palma              24.0               5.0            1982.0   Dona      1,276 â‚¬  True     True            Grup A      Analista           0.0    False        NaN                   NaN                        15.7


Con e DNI: 54013571W
- eliminar el registro 89 porque no tiene un salario coherente.

In [2305]:
df = df.drop(89)

In [2306]:
df_resultado = df[df['DNI'] == '55283701C']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

    Nom          Cognoms        DNI País d'origen Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional    Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
36  Ida  Karlsen Eriksen  55283701C       Noruega   Oslo               2.0               9.0            1992.0   Dona      1,614 â‚¬  True      NaN            Grup B  Analista           3.0      NaN        NaN                   NaN                        10.6
40  Ida  Karlsen Eriksen  55283701C       Noruega   Oslo               2.0               9.0            1992.0   Dona      1,614 â‚¬  True      NaN            Grup B  Analista           3.0      NaN        NaN                   NaN                        10.6


Con e DNI: 55283701C
- se puede eliminar cualquiera de los dos registros pue son identicos.eliminemos el registro 40

In [2307]:
df = df.drop(40)

In [2308]:
df_resultado = df[df['DNI'] == '55297384F']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

          Nom     Cognoms        DNI País d'origen    Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
730   Claudia  Gil Ferrer  55297384F       Espanya  Valencia               1.0              11.0            1981.0   Dona      1.789 â‚¬   NaN     True            Grup B  Responsable de vendes           0.0     True    29952.0                   9.2                        20.3
1003  Claudia  Gil Ferrer  55297384F       Espanya  Valencia               1.0              11.0            1981.0   Dona      1.789 â‚¬   NaN     True            Grup B  Responsable de vendes           0.0     True    29952.0                   9.2                        20.3


Con e DNI: 55297384F
- se puede eliminar cualquiera de los dos registros pues son identicos.

In [2309]:
df = df.drop(1003)

In [2310]:
df_resultado = df[df['DNI'] == '90585882A']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

         Nom      Cognoms        DNI País d'origen     Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
25      Sara        Serra  90585882A       Espanya  Tarragona              28.0               5.0            2002.0   Dona        993 â‚¬   NaN     True            Grup A          Administratiu           0.0    False        NaN                   NaN                        15.8
104    Oriol  Vila Casals  90585882A       Espanya     Madrid              30.0               9.0            1964.0   Home     1987 euros   NaN     True            Grup B  Responsable de vendes           0.0    False        NaN                   NaN                        13.5
114      Pol        Serra  90585882A       Espanya     Girona               4.0              12.0            1960.0   Home      1,178 â‚¬   NaN     True            Gr

Con e DNI: 90585882A
- se puede eliminar los registros 25,913 y dejar el registro 1000 poque es la misma persona y lo que se observa es un aumento de salario
-los registro 104,114, 281 Modificaremos los DNI por uno creado especial para no perder lo otros valores, en caso de obtener el DNI que le corresponda se actualizaria el DNI.

In [2311]:
df = df.drop([25,913])


In [2312]:
# 1. Diccionario con la estructura: 'Nombre': ('DNI_Viejo', 'DNI_Nuevo')
cambios_dni = {
    'Oriol': ('90585882A', '100000001A'),
    'Pol': ('90585882A', '100000002A'),
    'Xavier': ('90585882A', '100000003A')
}

# 2. Aplicar los cambios recorriendo el diccionario
for nombre, (dni_viejo, dni_nuevo) in cambios_dni.items():
    condicion = (df['Nom'] == nombre) & (df['DNI'] == dni_viejo)
    df['DNI'] = np.where(condicion, dni_nuevo, df['DNI'])

In [2316]:
df_resultado = df[df['DNI'] == '97954019T']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

       Nom Cognoms        DNI País d'origen  Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat
372   Alba   Muñoz  97954019T       Espanya  Bilbao              25.0              10.0            1983.0   Dona      1,409 â‚¬   NaN     True            Grup B  Responsable de vendes           0.0      NaN        NaN                   NaN                        14.7
1002  Alba   Muñoz  97954019T       Espanya  Bilbao              25.0              10.0            1983.0   Dona      1,409 â‚¬   NaN     True            Grup B  Responsable de vendes           0.0      NaN        NaN                   NaN                        14.7


Con e DNI: 97954019T
- se puede eliminar  cualquiera de los dos registros poque son identicos.por ejemplo el 1002

In [2317]:
df = df.drop(1002)

## Ejercicio 3 Transformaciones necesarias y preparación del dataset

 Limpieza inicial: Convertir a numérico y eliminar ceros

In [2318]:
columnas_fecha = ['Any de Naixement', 'Mes de Naixement', 'Dia de Naixement']
for col in columnas_fecha:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace(0, np.nan)


 Corregir MESES fuera de rango (Menores a 1 o mayores a 12, y nulos)
 Reemplazamos valores fuera de rango por NaN para unificar el tratamiento

In [2319]:
df.loc[(df['Mes de Naixement'] < 1) | (df['Mes de Naixement'] > 12), 'Mes de Naixement'] = np.nan

Asignamos un mes aleatorio entre 1 y 12 a los faltantes/erróneos para no acumular todo en Enero

In [2320]:
meses_validos = df['Mes de Naixement'].dropna()
if meses_validos.empty:
    meses_validos = pd.Series(range(1, 13))

df['Mes de Naixement'] = df['Mes de Naixement'].apply(lambda x: int(np.random.choice(meses_validos)) if pd.isna(x) else int(x))


Corregir AÑOS (Mismo método anterior para jubilados y menores)

In [2321]:
ANIO_ACTUAL = datetime.now().year
ANIO_MIN_MINIMO = ANIO_ACTUAL - 67  
ANIO_MAX_MAXIMO = ANIO_ACTUAL - 16  

anios_validos = df['Any de Naixement'].dropna()
anios_validos = anios_validos[(anios_validos >= ANIO_MIN_MINIMO) & (anios_validos <= ANIO_MAX_MAXIMO)]
if anios_validos.empty:
    anios_validos = pd.Series(range(ANIO_MIN_MINIMO, ANIO_MAX_MAXIMO + 1))

df['Any de Naixement'] = df['Any de Naixement'].apply(
    lambda x: int(np.random.choice(anios_validos)) if (pd.isna(x) or x < ANIO_MIN_MINIMO or x > ANIO_MAX_MAXIMO) else int(x)
)


Corregir DÍAS fuera de rango según el mes y año (Manejo de bisiestos)
Diccionario básico de días máximos por mes (Febrero se calcula dinámicamente después)
dias_por_mes = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}

Calcular el máximo de días para el mes actual
Si es bisiesto, Febrero tiene 29 días
Si el día supera el máximo permitido para ese mes, lo forzamos a 1

In [2322]:
def validar_dia(row):
    dia = row['Dia de Naixement']
    mes = row['Mes de Naixement']
    anio = row['Any de Naixement']
    
    if pd.isna(dia) or dia < 1:
        return 1
    dias_por_mes = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}
   
    max_dia = dias_por_mes.get(mes, 31)
    if mes == 2:
        
        if (anio % 4 == 0 and anio % 100 != 0) or (anio % 400 == 0):
            max_dia = 29
            
    
    if dia > max_dia:
        return 1
    return int(dia)

df['Dia de Naixement'] = df.apply(validar_dia, axis=1).astype(int)


Crear la fecha final validada 

In [2323]:
df['Fecha_Nacimiento_Validada'] = pd.to_datetime(dict(
    year=df['Any de Naixement'], 
    month=df['Mes de Naixement'], 
    day=df['Dia de Naixement']
), errors='coerce')


eliminar colunnas  originals con datos de fechas

In [2324]:
columnas_a_eliminar = ["Any de Naixement", "Mes de Naixement", "Dia de Naixement"]
df = df.drop(columns=columnas_a_eliminar, errors="ignore")

Analisis de las columnas relacionadas con los hijos

 Limpiar la columna numérica 'nombre de filis' rellenando con 0
  Si 'No Fills' es True o 'Fills' es False, asumimos que tiene 0 hijos de forma segura

In [2325]:
condicion_cero_hijos = (df["No Fills"] == True) | (df["Fills"] == False)
df.loc[condicion_cero_hijos, "Nombre_fills"] = df.loc[condicion_cero_hijos, "Nombre_fills"].fillna(0)

Crear la columna final unificada 'Fills'
Por defecto, tomamos el valor numérico que ya existe

In [2326]:
df["Fills_ok"] = df["Nombre_fills"]

 Rectificar los NaN restantes cruzando las columnas booleanas
Si 'Fills' es True pero el número de hijos es NaN, asumimos al menos 1 hijo 

In [2327]:
df.loc[(df["Fills"] == True) & (df["Nombre_fills"].isna()), "Fills"] = 1

Si todo sigue siendo NaN (no hay info en ninguna columna), asumimos 0 por defecto para limpiar el dataset

In [2328]:
df["Fills_ok"] = df["Fills_ok"].fillna(0)

Convertir a entero (int) para eliminar los decimales (.0) 

In [2329]:
df["Fills_ok"] = df["Fills_ok"].astype(int)


Eliminar las 3 columnas que ya no necesitamos para dejar el dataset limpio

In [2330]:
columnas_a_eliminar = ["Fills", "No Fills", "Nombre_fills"]
df = df.drop(columns=columnas_a_eliminar, errors="ignore")

Analisis de datos coches

Si hay valores negativos por error al transcribir lo convertimos a positivo positivo

In [2331]:
df["Km_anuals"] = df["Km_anuals"].abs()

Si 'Km_anuals' o 'Consum_mitja_L_100km' no son vacios entonces gaarantizamos que tiene coche

In [2332]:
tiene_datos_coche = df['Km_anuals'].notna() | df['Consum_mitja_L_100km'].notna()
df.loc[tiene_datos_coche, 'Te_cotxe'] = True


Resolución de incoherencia (Consumo sin Kilometraje): Imputamos el Km mediante la mediana 
de los que sí tienen coche para no alterar la distribución con valores extremos (outliers)

In [2333]:
incoherencia_km = df['Consum_mitja_L_100km'].notna() & df['Km_anuals'].isna()
mediana_km = df[df['Te_cotxe'] == True]['Km_anuals'].median()
df.loc[incoherencia_km, 'Km_anuals'] = round(mediana_km,1)


 Resolución de incoherencia (Kilometraje sin consumo): Imputamos el consumo mediante la mediana 
 de los que sí tienen coche para no alterar la distribución con valores extremos (outliers)

In [2334]:
incoherencia_consum_L = df['Consum_mitja_L_100km'].isna() & df['Km_anuals'].notna()
mediana_consum_L = df[df['Te_cotxe'] == True]['Consum_mitja_L_100km'].median()
df.loc[incoherencia_consum_L, 'Consum_mitja_L_100km'] = round(mediana_consum_L,1)

Si no tiene datos de coche en absoluto, aseguramos que conste como 'False' y sus métricas sean 0

In [2335]:
df.loc[df['Te_cotxe'].isna(), 'Te_cotxe'] = False
df.loc[df['Te_cotxe'] == False, ['Km_anuals', 'Consum_mitja_L_100km']] = 0

Identificamos a las personas que tienen 'Te_cotxe' == TRUE pero sus métricas están vacías

In [2336]:
incoherencia_tiene_coche = (df['Te_cotxe'] == True) & (df['Km_anuals'].isna()) & (df['Consum_mitja_L_100km'].isna())

Calculamos las medianas de referencia del grupo que SÍ tiene coche y SÍ tiene datos válidos

In [2337]:
mediana_km_con_coche = df[(df['Te_cotxe'] == True) & (df['Km_anuals'] > 0)]['Km_anuals'].median()
mediana_consumo_con_coche = df[(df['Te_cotxe'] == True) & (df['Consum_mitja_L_100km'] > 0)]['Consum_mitja_L_100km'].median()

Asignamos estas medianas solo a los registros que presentaban la incoherencia

In [2338]:
df.loc[incoherencia_tiene_coche, 'Km_anuals'] = mediana_km_con_coche
df.loc[incoherencia_tiene_coche, 'Consum_mitja_L_100km'] = mediana_consumo_con_coche

La Temperatura

 IMPUTACIÓN DE TEMPERATURA MEDIA

Agrupamos por 'Ciutat' para obtener su temperatura media. Si la ciudad no tiene datos, 
usamos una jerarquía superior: la media global del dataset.

In [2339]:

if 'Temperatura_mitjana_ciutat' in df.columns:
    media_por_ciudad = df.groupby('Ciutat')['Temperatura_mitjana_ciutat'].transform('mean').round(1)
    media_global_temp = round(df['Temperatura_mitjana_ciutat'].mean(),1)
    
    df['Temperatura_mitjana_ciutat'] = df['Temperatura_mitjana_ciutat'].fillna(media_por_ciudad).fillna(media_global_temp)

IMPUTACIÓN DE SALARIOS FALTANTES

Eliminar todo lo que no sea númerico 

In [2340]:
df['Salari mensual'] = df['Salari mensual'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df['Salari mensual'] = pd.to_numeric(df['Salari mensual'], errors='coerce').round(2)

El salario depende estrechamente del Cargo ('Carrec') y de la 'Ciutat'. 
Calculamos la mediana de estas agrupaciones para evitar que los sueldos extremadamente altos distorsionen la imputación.

 LIMPIEZA: Convertir salarios sospechosamente bajos (ej. < 600) en NaN, para que no afectaen mis calculos de las medianas
 Calcular las medianas limpias 
 Imputación en cascada

In [2341]:

if 'Salari mensual' in df.columns:
        
    UMBRAL_MINIMO = 600
    df.loc[df['Salari mensual'] < UMBRAL_MINIMO, 'Salari mensual'] = None
    
    
    mediana_compleja = df.groupby(['Carrec', 'Ciutat'])['Salari mensual'].transform('median')
    mediana_por_cargo = df.groupby('Carrec')['Salari mensual'].transform('median')
    mediana_global = df['Salari mensual'].median()
    

    df['Salari mensual'] = (df['Salari mensual'].fillna(mediana_compleja).fillna(mediana_por_cargo).fillna(mediana_global))

In [2342]:
display (df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,Fecha_Nacimiento_Validada,Fills_ok
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1469.0,Grup B,Cap de projecte,True,32108.0,25.0,10.1,1998-03-23,3
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2718.0,Grup C,Senior analyst,True,19496.0,10.4,18.7,1960-11-08,0
2,Noa,Serra,14308421X,Espanya,Alacant,Home,1358.0,Grup A,Tecnic IT,False,0.0,0.0,16.7,1961-04-27,4
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1478.0,Grup B,Data Analyst,False,0.0,0.0,18.3,1985-10-13,2
4,David,Vila,82070937P,Espanya,Bilbao,Home,1284.0,Grup B,Administratiu,True,56135.0,11.8,13.1,1965-11-29,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Home,1074.0,Grup A,Data Analyst,True,80000.0,12.0,12.0,1961-10-12,1
998,Chloé,Dubois,66354268T,Francia,Nice,Home,1954.0,Grup B,Data Analyst,False,0.0,0.0,11.7,1998-12-02,1
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,Home,2010.0,Grup C,Analista,False,0.0,0.0,-10.0,1991-12-31,1
1000,Sara,Serra,90585882A,Espanya,Tarragona,Dona,1073.0,Grup A,Administratiu,False,0.0,0.0,15.8,2002-05-28,0


## Ejercio 4 Primeros análisis descriptivos

Tabla resumen descriptiva

In [2343]:

tabla_genero = df.groupby('Gènere')['Salari mensual'].agg(
    Sueldo_Medio='mean',
    Mediana='median',
    Mínimo='min',
    Máximo='max'
).sort_values(by='Sueldo_Medio', ascending=False)

print("--- Resumen de Salarios por Género ---")
print(tabla_genero.round(2))


--- Resumen de Salarios por Género ---
        Sueldo_Medio  Mediana  Mínimo  Máximo
Gènere                                       
Home         1531.48   1377.0   673.0  3672.0
Dona         1530.61   1393.0   722.0  3672.0


Tabla pivote con los márgenes (totales/promedios globales)

In [2344]:
matriz_salario = df.pivot_table(
    values='Salari mensual', 
    index='Gènere', 
    columns="País d'origen", 
    aggfunc='mean', 
    margins=True, 
    margins_name='Media Global'
)


Aplicar formato condicional con un mapa de color (gradiente)
Esto se visualizará con colores intensos en los valores altos

In [2345]:
matriz_estilizada = matriz_salario.style.background_gradient(
    cmap='YlOrRd', 
    subset=(matriz_salario.index[:-1], matriz_salario.columns[:-1]) 
).format("{:.2f}")


matriz_estilizada

País d'origen,Alemanya,Espanya,Francia,Itàlia,Noruega,Media Global
Gènere,,,,,,
Dona,1462.28,1540.02,1391.76,1499.55,1843.85,1530.61
Home,1643.33,1527.54,1532.44,1594.63,1384.76,1531.48
Media Global,1530.18,1533.20,1470.28,1539.17,1598.29,1531.07


# Nivel 2 - Análisis con datos enriquecidos y estructuras múltiples

## Ejercicio 1 Enriquecimiento mediante datasets externos

Cargar ficheros

In [2346]:
df_ciutats = pd.read_csv('E:/osley/Barecelona activa/curso especializacion analisis de datos/sprint 8/ciutats_context.csv', encoding='cp1252')

Convertimos los códigos Unicode de texto plano (como \u00f1) a caracteres reales
Por seguridad, eliminamos cualquier espacio en blanco oculto que pueda romper el merge

In [2347]:
df_ciutats['Ciutat'] = df_ciutats['Ciutat'].str.encode('utf-8').str.decode('unicode-escape')

df_ciutats['Ciutat'] = df_ciutats['Ciutat'].str.strip()

In [2348]:
mapeo_caracteres = {
    'Ã\xa0': 'à', 'Ã©': 'é', 'Ã\xad': 'í', 'Ãº': 'ú', 'Ã‰': 'Á', 'ÃÂ¨':'è',
    'Ã±': 'ñ', 'Ã³': 'ó', 'Ã¡': 'á', 'Ã¨': 'e', 'Ã\x81': 'A', 
    'Ã¤': 'ä', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã§': 'ci', 'Ã¸': 'ø'
}


In [2349]:


def corregir_subcadenas_campo(dataframe, columnas_a_corregir, diccionario_cambios):
    for col in columnas_a_corregir:
        if col in dataframe.columns:
            
            dataframe[col] = (dataframe[col]
                              .astype(str)
                              .str.replace(diccionario_cambios, regex=True))


In [2350]:
lista_campos = ['Ciutat']
corregir_subcadenas_campo(df_ciutats, lista_campos, mapeo_caracteres)

In [2351]:
display(df_ciutats)

,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus
0,Barcelona,Espanya,78,8.5,Mediterrani
1,Madrid,Espanya,75,8.0,Continental
2,Valencia,Espanya,70,7.5,Mediterrani
3,Sevilla,Espanya,65,7.0,Mediterrani
4,Bilbao,Espanya,72,7.8,Oceànic
5,Zaragoza,Espanya,68,7.0,Continental
6,Murcia,Espanya,60,6.5,Mediterrani
7,Palma,Espanya,73,7.2,Mediterrani
8,Girona,Espanya,66,6.8,Mediterrani
9,Tarragona,Espanya,67,6.9,Mediterrani


In [2352]:
ruta_archivo = 'E:/osley/Barecelona activa/curso especializacion analisis de datos/sprint 8/matriu_distancies.xlsx'

In [2353]:

with open('politica_salarial.json', 'r') as archivo:
    p_salarial = json.load(archivo)

    print(p_salarial)

{'base_increment_by_group': {'Grup A': 5.0, 'Grup B': 3.5, 'Grup C': 2.0, 'Grup D': 8.0}, 'children_adjustment': {'has_children': 1.5, 'no_children': 0.0}, 'age_adjustment': {'18-29': 0.0, '30-44': 1.0, '45-59': 2.0, '60+': 1.5}, 'cost_of_living_adjustment': {'low': 0.0, 'medium': 1.0, 'high': 2.0}, 'transport_adjustment': {'low': 0.5, 'medium': 0.0, 'high': -0.5}, 'rules': {'max_total_increment_percent': 15, 'min_salary_threshold': 800}}


In [2354]:
display(df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,Fecha_Nacimiento_Validada,Fills_ok
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1469.0,Grup B,Cap de projecte,True,32108.0,25.0,10.1,1998-03-23,3
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2718.0,Grup C,Senior analyst,True,19496.0,10.4,18.7,1960-11-08,0
2,Noa,Serra,14308421X,Espanya,Alacant,Home,1358.0,Grup A,Tecnic IT,False,0.0,0.0,16.7,1961-04-27,4
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1478.0,Grup B,Data Analyst,False,0.0,0.0,18.3,1985-10-13,2
4,David,Vila,82070937P,Espanya,Bilbao,Home,1284.0,Grup B,Administratiu,True,56135.0,11.8,13.1,1965-11-29,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Home,1074.0,Grup A,Data Analyst,True,80000.0,12.0,12.0,1961-10-12,1
998,Chloé,Dubois,66354268T,Francia,Nice,Home,1954.0,Grup B,Data Analyst,False,0.0,0.0,11.7,1998-12-02,1
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,Home,2010.0,Grup C,Analista,False,0.0,0.0,-10.0,1991-12-31,1
1000,Sara,Serra,90585882A,Espanya,Tarragona,Dona,1073.0,Grup A,Administratiu,False,0.0,0.0,15.8,2002-05-28,0


Mapeo de Grupo Profesional con el diccionario p_salarial

In [2355]:
df['Incremento_Grupo_Profesional'] = df['Grup Professional'].map(p_salarial['base_increment_by_group']).fillna(0.0)

Para hijos
Extraemos los valores del diccionario 
Si Fills_ok es mayor que 0 asigna 'valor_con_hijos', si es 0 se asigna 'valor_sin_hijos'

In [2356]:
valor_con_hijos = p_salarial['children_adjustment']['has_children']
valor_sin_hijos = p_salarial['children_adjustment']['no_children']

df['Incremento_por_hijos'] = np.where(df['Fills_ok'] > 0, valor_con_hijos, valor_sin_hijos)

Para edad
Calcular la edad actual basada en el año 2026
Dividimos los días transcurridos entre 365.25 (para contar los años bisiestos)

In [2357]:
edades = ((pd.Timestamp.now() - df['Fecha_Nacimiento_Validada']).dt.days / 365.25).round(2)

Definimos los límites numéricos y las etiquetas del diccionario

In [2358]:
limites = [18, 30, 45, 60, 130]
etiquetas = ['18-29', '30-44', '45-59', '60+']

Clasificar las edades en sus rangos correspondientes
right=False hace que los límites sean incluyentes a la izquierda (ej: el 30 entra en '30-44')

In [2359]:
df['Franja_Edad'] = pd.cut(edades, bins=limites, labels=etiquetas, right=False)

Mapear los rangos de texto con los valores de tu diccionario

In [2360]:
df['Incremento_Franja_Edad'] = df['Franja_Edad'].map(p_salarial['age_adjustment']).fillna(0.0)

 Definir los límites numéricos y las etiquetas de texto

In [2361]:
limites_coste = [0, 70, 80, 100] #np.inf
etiquetas_coste = ['low', 'medium', 'high']

Crear la columna de categoría (Se guarda en 'Coste_Vida_Categoria')
right=False asegura que: menos de 70 es 'low', 70-79 es 'medium', 80+ es 'high'

In [2362]:
df_ciutats['Coste_Vida_Categoria'] = pd.cut(
    df_ciutats['Cost_vida_index'], 
    bins=limites_coste, 
    labels=etiquetas_coste, 
    right=False
)

In [2363]:
display(df_ciutats)

,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus,Coste_Vida_Categoria
0,Barcelona,Espanya,78,8.5,Mediterrani,medium
1,Madrid,Espanya,75,8.0,Continental,medium
2,Valencia,Espanya,70,7.5,Mediterrani,medium
3,Sevilla,Espanya,65,7.0,Mediterrani,low
4,Bilbao,Espanya,72,7.8,Oceànic,medium
5,Zaragoza,Espanya,68,7.0,Continental,low
6,Murcia,Espanya,60,6.5,Mediterrani,low
7,Palma,Espanya,73,7.2,Mediterrani,medium
8,Girona,Espanya,66,6.8,Mediterrani,low
9,Tarragona,Espanya,67,6.9,Mediterrani,low


 Mapear las categorías con el diccionario usando la columna recién creada

In [2364]:
df_ciutats['Ajuste_Coste_Vida'] = df_ciutats['Coste_Vida_Categoria'].map(p_salarial['cost_of_living_adjustment']).fillna(0.0)


In [2365]:
display(df_ciutats)

,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus,Coste_Vida_Categoria,Ajuste_Coste_Vida
0,Barcelona,Espanya,78,8.5,Mediterrani,medium,1.0
1,Madrid,Espanya,75,8.0,Continental,medium,1.0
2,Valencia,Espanya,70,7.5,Mediterrani,medium,1.0
3,Sevilla,Espanya,65,7.0,Mediterrani,low,0.0
4,Bilbao,Espanya,72,7.8,Oceànic,medium,1.0
5,Zaragoza,Espanya,68,7.0,Continental,low,0.0
6,Murcia,Espanya,60,6.5,Mediterrani,low,0.0
7,Palma,Espanya,73,7.2,Mediterrani,medium,1.0
8,Girona,Espanya,66,6.8,Mediterrani,low,0.0
9,Tarragona,Espanya,67,6.9,Mediterrani,low,0.0


Si la columna ya existe, la eliminamos para que no se duplique al fusionar

In [2366]:
if 'Coste_Vida_Categoria_y' in df.columns:
    df = df.drop(columns=['Coste_Vida_Categoria_y'])

In [2367]:
# 1. Eliminamos los duplicados de la tabla de ciudades ANTES del merge
# Nos quedamos solo con la primera aparición de cada ciudad
df_ciutats_limpio = df_ciutats[['Ciutat', 'Coste_Vida_Categoria', 'Ajuste_Coste_Vida']].drop_duplicates(subset=['Ciutat'])

# 2. Hacemos el merge con la tabla limpia (AHORA YA NO HABRÁ DUPLICADOS)
df = df.merge(
    df_ciutats_limpio, 
    on='Ciutat', 
    how='left'
)

Rellenar con 0.0 si alguna ciudad del empleado no se encontró en la tabla de costes

In [2368]:
df['Ajuste_Coste_Vida'] = df['Ajuste_Coste_Vida'].fillna(0.0)

In [2369]:
display(df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,...,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,Fecha_Nacimiento_Validada,Fills_ok,Incremento_Grupo_Profesional,Incremento_por_hijos,Franja_Edad,Incremento_Franja_Edad,Coste_Vida_Categoria,Ajuste_Coste_Vida
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1469.0,Grup B,Cap de projecte,True,...,25.0,10.1,1998-03-23,3,3.5,1.5,18-29,0.0,low,0.0
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2718.0,Grup C,Senior analyst,True,...,10.4,18.7,1960-11-08,0,2.0,0.0,60+,1.5,low,0.0
2,Noa,Serra,14308421X,Espanya,Alacant,Home,1358.0,Grup A,Tecnic IT,False,...,0.0,16.7,1961-04-27,4,5.0,1.5,60+,1.5,low,0.0
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1478.0,Grup B,Data Analyst,False,...,0.0,18.3,1985-10-13,2,3.5,1.5,30-44,1.0,low,0.0
4,David,Vila,82070937P,Espanya,Bilbao,Home,1284.0,Grup B,Administratiu,True,...,11.8,13.1,1965-11-29,0,3.5,0.0,60+,1.5,medium,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Home,1074.0,Grup A,Data Analyst,True,...,12.0,12.0,1961-10-12,1,5.0,1.5,60+,1.5,low,0.0
989,Chloé,Dubois,66354268T,Francia,Nice,Home,1954.0,Grup B,Data Analyst,False,...,0.0,11.7,1998-12-02,1,3.5,1.5,18-29,0.0,NaN,0.0
990,Adrià,Vila Díaz,57511543T,Espanya,Palma,Home,2010.0,Grup C,Analista,False,...,0.0,-10.0,1991-12-31,1,2.0,1.5,30-44,1.0,medium,1.0
991,Sara,Serra,90585882A,Espanya,Tarragona,Dona,1073.0,Grup A,Administratiu,False,...,0.0,15.8,2002-05-28,0,5.0,0.0,18-29,0.0,low,0.0


Inicializar la columna de problemas como texto vacío

In [2371]:
df['Problemas_Ajuste'] = ''

Detectar problemas de forma vectorizada 
Grupo Profesional no encontrado (Busca en las llaves del diccionario)

In [2372]:
mask_grupo = ~df['Grup Professional'].isin(p_salarial['base_increment_by_group'])
df.loc[mask_grupo, 'Problemas_Ajuste'] += 'Grupo Profesional no definido en política; '

Situación de no tener valores de hijos no encontrada 

In [2373]:
mask_familiar = df['Fills_ok'].isna()
df.loc[mask_familiar, 'Problemas_Ajuste'] += 'Dato de hijos inválido o no definido; '

Franja de Edad no encontrada

In [2374]:
mask_edad = df['Franja_Edad'].isna()
df.loc[mask_edad, 'Problemas_Ajuste'] += 'Edad fuera de rango o fecha de nacimiento inválida; '

Ciudad no encontrada 
(Si la fusión de tablas falló porque la ciudad del empleado no existe en df_ciutats)

In [2375]:
mask_ciudad_none = df['Coste_Vida_Categoria'].isna()
df.loc[mask_ciudad_none, 'Problemas_Ajuste'] += 'Ciudad no encontrada'

Asegurar tipo de datos flotante para evitar el TypeError

In [2376]:
df['Incremento_Franja_Edad'] = df['Incremento_Franja_Edad'].astype(float)
df['Ajuste_Coste_Vida'] = df['Ajuste_Coste_Vida'].astype(float)

 Cálculos finales del salario

In [2377]:
df['Incremento_Total_Porcentaje'] = (
    df['Incremento_Grupo_Profesional'] +
    df['Incremento_por_hijos'] +
    df['Incremento_Franja_Edad'] +
    df['Ajuste_Coste_Vida']
)

Aplicando el salario ya ajustado

In [2378]:
df['Salario_Ajustado'] = (df['Salari mensual'] * ((1 + df['Incremento_Total_Porcentaje']/100))).round(2)

display(df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,...,Fills_ok,Incremento_Grupo_Profesional,Incremento_por_hijos,Franja_Edad,Incremento_Franja_Edad,Coste_Vida_Categoria,Ajuste_Coste_Vida,Problemas_Ajuste,Incremento_Total_Porcentaje,Salario_Ajustado
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1469.0,Grup B,Cap de projecte,True,...,3,3.5,1.5,18-29,0.0,low,0.0,,5.0,1542.45
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2718.0,Grup C,Senior analyst,True,...,0,2.0,0.0,60+,1.5,low,0.0,,3.5,2813.13
2,Noa,Serra,14308421X,Espanya,Alacant,Home,1358.0,Grup A,Tecnic IT,False,...,4,5.0,1.5,60+,1.5,low,0.0,,8.0,1466.64
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1478.0,Grup B,Data Analyst,False,...,2,3.5,1.5,30-44,1.0,low,0.0,,6.0,1566.68
4,David,Vila,82070937P,Espanya,Bilbao,Home,1284.0,Grup B,Administratiu,True,...,0,3.5,0.0,60+,1.5,medium,1.0,,6.0,1361.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Home,1074.0,Grup A,Data Analyst,True,...,1,5.0,1.5,60+,1.5,low,0.0,,8.0,1159.92
989,Chloé,Dubois,66354268T,Francia,Nice,Home,1954.0,Grup B,Data Analyst,False,...,1,3.5,1.5,18-29,0.0,NaN,0.0,Ciudad no encontrada,5.0,2051.70
990,Adrià,Vila Díaz,57511543T,Espanya,Palma,Home,2010.0,Grup C,Analista,False,...,1,2.0,1.5,30-44,1.0,medium,1.0,,5.5,2120.55
991,Sara,Serra,90585882A,Espanya,Tarragona,Dona,1073.0,Grup A,Administratiu,False,...,0,5.0,0.0,18-29,0.0,low,0.0,,5.0,1126.65


In [2379]:
df_filtrado_ajuste = df[df['Problemas_Ajuste'] != '']
display(df_filtrado_ajuste)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,...,Fills_ok,Incremento_Grupo_Profesional,Incremento_por_hijos,Franja_Edad,Incremento_Franja_Edad,Coste_Vida_Categoria,Ajuste_Coste_Vida,Problemas_Ajuste,Incremento_Total_Porcentaje,Salario_Ajustado
8,Nora,Andersen Pedersen,37289414J,Noruega,Stavanger,Dona,1338.0,Grup B,Analista junior,False,...,0,3.5,0.0,60+,1.5,NaN,0.0,Ciudad no encontrada,5.0,1404.90
21,Sergio,Ribera,30509175E,Espanya,Màlaga,Home,1880.0,Grup B,Senior analyst,False,...,4,3.5,1.5,45-59,2.0,NaN,0.0,Ciudad no encontrada,7.0,2011.60
26,Chloé,Durand,96691619E,Francia,Toulouse,Home,751.0,Grup A,Responsable de vendes,False,...,2,5.0,1.5,45-59,2.0,NaN,0.0,Ciudad no encontrada,8.5,814.83
34,Hugo,Pujol,83715012G,Espanya,Màlaga,Home,1974.0,Grup B,Cap de projecte,False,...,0,3.5,0.0,60+,1.5,NaN,0.0,Ciudad no encontrada,5.0,2072.70
50,Marina,Vidal,24524825B,Espanya,Màlaga,Dona,1647.0,Grup B,Administratiu,False,...,0,3.5,0.0,30-44,1.0,NaN,0.0,Ciudad no encontrada,4.5,1721.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,Marie,Becker,86737279H,Alemanya,Dusseldorf,Dona,1421.0,Grup B,Analista junior,False,...,0,3.5,0.0,45-59,2.0,NaN,0.0,Ciudad no encontrada,5.5,1499.16
950,Jonas,Muller Fischer,74877175D,Alemanya,Köln,Dona,1501.0,Grup B,Data Analyst,False,...,4,3.5,1.5,45-59,2.0,NaN,0.0,Ciudad no encontrada,7.0,1606.07
969,Giulia,Conti Russo,72192924S,Itàlia,Bologna,Dona,1931.0,Grup C,Administratiu,False,...,0,2.0,0.0,30-44,1.0,NaN,0.0,Ciudad no encontrada,3.0,1988.93
986,Laura,Bauer Zimmermann,34637932B,Alemanya,Köln,Home,1644.0,Grup B,Analista,False,...,0,3.5,0.0,30-44,1.0,NaN,0.0,Ciudad no encontrada,4.5,1717.98


Estos casos de ciudades no encontradas que no tienen costo de vida , afectan porque no se tiene por tanto una clasificacion para darle una un valos de porcentaje para sumar al al incremento de salario,
Una solucion podria ser tomar la mediana de las ciudades que perentencen al mismo país y asignarle el valor deacuerdo al rengo que encaje, si es una ciudad que el pais solo esta representado por esta ciudad se toamria la mediana general del datset.

## Ejercicio 2 Reestructuración del dataset

 Reestructurar el dataset utilizando melt() para convertirlo a formato largo, definiendo previamente:
 -las columnas que actúan como identificadoras (por ejemplo: persona, ciudad, grupo profesional, género, etc.).
 -las columnas que pasan a representar valores medidos (por ejemplo: salarios, movilidad, variables contextuales).
 Analizar por qué el formato largo es más adecuado para:
 -realizar comparaciones entre distintos tipos de variables.
 -calcular estadísticos agregados.

 Definir las columnas fijas que identifican al empleado

In [2380]:
columnas_identificadoras = [
    'DNI', 'Nom', 'Cognoms', "País d'origen", 'Ciutat', 
    'Gènere', 'Grup Professional', 'Carrec', 'Fecha_Nacimiento_Validada', 
    'Franja_Edad', 'Fills_ok', 'Problemas_Ajuste'
]

 Definir las columnas representan valores medidos,  agrupadas por su concepto teórico
--- Bloque Salarios ---
--- Bloque Movilidad ---
--- Bloque Contexto Climático y Ciudad ---

In [2381]:
columnas_valores = [

    'Salari mensual', 
    'Salario_Ajustado',
    
    'Km_anuals', 
    'Consum_mitja_L_100km',
    
    'Temperatura_mitjana_ciutat'
]

Aplicar melt() para pivotar el dataset a formato largo
'Concepto_Metrica' Guardará etiquetas como 'Salari mensual', 'Km_anuals', etc.
'Valor_Medido'  Guardará el número flotante o entero correspondiente

In [2382]:
df_largo = pd.melt(
    df,
    id_vars=columnas_identificadoras,
    value_vars=columnas_valores,
    var_name='Concepto_Metrica',  
    value_name='Valor_Medido'     
)

Verificar la transformación

In [2383]:
print(f"Dimensiones del dataset original (Ancho): {df.shape}")
print(f"Dimensiones del dataset nuevo (Largo): {df_largo.shape}")
display(df_largo.head(15))

Dimensiones del dataset original (Ancho): (993, 24)
Dimensiones del dataset nuevo (Largo): (4965, 14)


,DNI,Nom,Cognoms,País d'origen,Ciutat,Gènere,Grup Professional,Carrec,Fecha_Nacimiento_Validada,Franja_Edad,Fills_ok,Problemas_Ajuste,Concepto_Metrica,Valor_Medido
0,66722344X,Joana,Gil Navarro,Espanya,Valladolid,Dona,Grup B,Cap de projecte,1998-03-23,18-29,3,,Salari mensual,1469.0
1,48840994W,Marc,Muñoz,Espanya,Alacant,Home,Grup C,Senior analyst,1960-11-08,60+,0,,Salari mensual,2718.0
2,14308421X,Noa,Serra,Espanya,Alacant,Home,Grup A,Tecnic IT,1961-04-27,60+,4,,Salari mensual,1358.0
3,58586340F,Pol,Gil,Espanya,Sevilla,Home,Grup B,Data Analyst,1985-10-13,30-44,2,,Salari mensual,1478.0
4,82070937P,David,Vila,Espanya,Bilbao,Home,Grup B,Administratiu,1965-11-29,60+,0,,Salari mensual,1284.0
5,25014631B,Alexia,Puig,Espanya,Las Palmas de Gran Canaria,Dona,Grup B,Cap de projecte,1978-08-01,45-59,0,,Salari mensual,2069.0
6,36998038E,David,Gómez Vila,Espanya,Palma,Home,Grup B,Cap de projecte,1971-09-25,45-59,0,,Salari mensual,2349.0
7,47376585K,Carmen,López,Espanya,Barcelona,Dona,Grup C,Responsable de vendes,1962-09-08,60+,4,,Salari mensual,2855.0
8,37289414J,Nora,Andersen Pedersen,Noruega,Stavanger,Dona,Grup B,Analista junior,1963-03-08,60+,0,Ciudad no encontrada,Salari mensual,1338.0
9,75181648S,Alba,García,Espanya,Las Palmas de Gran Canaria,Dona,Grup A,Senior analyst,1995-10-01,30-44,4,,Salari mensual,1450.0


Calcular estadísticos descriptivos agregados en formato largo

In [2384]:
df_agregado = df_largo.groupby(['Gènere', 'Grup Professional', 'Concepto_Metrica'])['Valor_Medido'].agg(['mean', 'std']).reset_index()

Renombrar las columnas para que el reporte sea limpio y legible

In [2385]:
df_agregado.columns = ['Gènere', 'Grup Professional', 'Métrica', 'Promedio', 'Desviación Estándar']

Redondear los resultados a 2 decimales para mantener la limpieza visual

In [2386]:
df_agregado = df_agregado.round(2)

 Mostrar el resultado completo ordenado

In [2387]:
display(df_agregado.sort_values(by=['Métrica', 'Gènere']))

,Gènere,Grup Professional,Métrica,Promedio,Desviación Estándar
0,Dona,Grup A,Consum_mitja_L_100km,3.46,7.08
5,Dona,Grup B,Consum_mitja_L_100km,4.20,7.37
10,Dona,Grup C,Consum_mitja_L_100km,3.51,6.23
15,Dona,Grup D,Consum_mitja_L_100km,3.68,6.91
20,Home,Grup A,Consum_mitja_L_100km,3.78,6.73
25,Home,Grup B,Consum_mitja_L_100km,3.03,6.07
30,Home,Grup C,Consum_mitja_L_100km,2.65,6.16
35,Home,Grup D,Consum_mitja_L_100km,5.71,8.95
1,Dona,Grup A,Km_anuals,13763.28,27417.82
6,Dona,Grup B,Km_anuals,19596.66,31745.76


## Ejercicio 3  Creación de variables derivadas (Feature Engineering)

 VARIABLE 1: Índice de Carga Familiar (ICF)
Evitamos la división por cero sumando 1 al número de hijos(por ejemplo para las personas que no tiene hijos)

In [2388]:
df['Indice_Carga_Familiar'] = (df['Salari mensual'] / (df['Fills_ok'] + 1)).round(2)

VARIABLE 2: Ratio de Poder Adquisitivo Local (RPAL)
Relaciona el salario ajustado con el índice de costo de vida numérico original

 Asegurar que cada ciudad aparezca SOLO UNA VEZ en df_ciutats
 Hacer el merge limpio

In [ ]:

df_ciutats_unico = df_ciutats[['Ciutat', 'Cost_vida_index']].drop_duplicates(subset=['Ciutat'])
df['Ciutat'] = df['Ciutat'].astype(str).str.strip()

df = df.merge(df_ciutats_unico, on='Ciutat', how='left')

In [ ]:
df['Ratio_Poder_Adquisitivo'] = (df['Salario_Ajustado'] / df['Cost_vida_index']).round(2)

 VARIABLE 3: Indicador de Etapa Vital y Carga (Categoría)

Calculamos la edad

In [2391]:
df['Edad'] = ((pd.Timestamp.now() - df['Fecha_Nacimiento_Validada']).dt.days / 365.25).astype(int)

 0 -> Joven/Adulto sin cargas (Edad < 45 y 0 hijos)
 1 -> Etapa Activa con cargas (Cualquier edad con hijos, o Edad >= 45 sin hijos)
 2 -> Alta Dependencia Próxima a Retiro (Edad >= 60 con hijos)
 El caso por defecto (Else) captura la "Etapa Activa con cargas" (Tipo 1)

In [2392]:
condiciones = [
    (df['Edad'] < 45) & (df['Fills_ok'] == 0),  # Tipo 0
    (df['Edad'] >= 60) & (df['Fills_ok'] > 0),   # Tipo 2 (Priorizamos el caso extremo)
]

df['Indicador_Etapa_Vital'] = np.select(condiciones, [0, 2], default=1)

 --- DETECCIÓN AUTOMÁTICA DE CASOS LÍMITE O INCOHERENTES ---

 Caso Límite 1: Valores infinitos o indeterminados por nulos

In [2393]:
df['Alertas_Calidad_Feature'] = ''

mask_inf_rpal = np.isinf(df['Ratio_Poder_Adquisitivo']) | df['Ratio_Poder_Adquisitivo'].isna()
df.loc[mask_inf_rpal, 'Alertas_Calidad_Feature'] += 'Costo de vida inválido o cero (RPAL Infinito); '

 Caso Límite 2: Salarios extremadamente bajos compartidos con alta carga familiar

In [2394]:
mask_vulnerabilidad = (df['Indice_Carga_Familiar'] < 500) & (df['Fills_ok'] > 0)
df.loc[mask_vulnerabilidad, 'Alertas_Calidad_Feature'] += 'Alta vulnerabilidad financiera (ICF < 500); '

Caso Límite 3: Incoherencias biológicas (Menores de edad registrados o edades imposibles)

In [2395]:
mask_edad_incoherente = (df['Edad'] < 18) | (df['Edad'] > 80)
df.loc[mask_edad_incoherente, 'Alertas_Calidad_Feature'] += 'Revisar Edad (Fuera de rango laboral 18-80); '

Visualizar las nuevas variables y sus alertas si existen

In [2396]:
columnas_verificacion = ['Nom', 'Salari mensual', 'Fills_ok', 'Edad', 'Indice_Carga_Familiar', 'Ratio_Poder_Adquisitivo', 'Indicador_Etapa_Vital', 'Alertas_Calidad_Feature']
display(df[columnas_verificacion].head(10))

,Nom,Salari mensual,Fills_ok,Edad,Indice_Carga_Familiar,Ratio_Poder_Adquisitivo,Indicador_Etapa_Vital,Alertas_Calidad_Feature
0,Joana,1469.0,3,28,367.25,24.10,1,Alta vulnerabilidad financiera (ICF < 500);
1,Marc,2718.0,0,65,2718.00,41.37,1,
2,Noa,1358.0,4,65,271.60,21.57,2,Alta vulnerabilidad financiera (ICF < 500);
3,Pol,1478.0,2,40,492.67,24.10,1,Alta vulnerabilidad financiera (ICF < 500);
4,David,1284.0,0,60,1284.00,18.90,1,
5,Alexia,2069.0,0,47,2069.00,35.21,1,
6,David,2349.0,0,54,2349.00,34.27,1,
7,Carmen,2855.0,4,63,571.00,38.80,2,
8,Nora,1338.0,0,63,1338.00,NaN,1,Costo de vida inválido o cero (RPAL Infinito);
9,Alba,1450.0,4,30,290.00,25.14,1,Alta vulnerabilidad financiera (ICF < 500);


In [2397]:
display(df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,...,Ajuste_Coste_Vida,Problemas_Ajuste,Incremento_Total_Porcentaje,Salario_Ajustado,Indice_Carga_Familiar,Cost_vida_index,Ratio_Poder_Adquisitivo,Edad,Indicador_Etapa_Vital,Alertas_Calidad_Feature
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1469.0,Grup B,Cap de projecte,True,...,0.0,,5.0,1542.45,367.25,64.0,24.10,28,1,Alta vulnerabilidad financiera (ICF < 500);
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2718.0,Grup C,Senior analyst,True,...,0.0,,3.5,2813.13,2718.00,68.0,41.37,65,1,
2,Noa,Serra,14308421X,Espanya,Alacant,Home,1358.0,Grup A,Tecnic IT,False,...,0.0,,8.0,1466.64,271.60,68.0,21.57,65,2,Alta vulnerabilidad financiera (ICF < 500);
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1478.0,Grup B,Data Analyst,False,...,0.0,,6.0,1566.68,492.67,65.0,24.10,40,1,Alta vulnerabilidad financiera (ICF < 500);
4,David,Vila,82070937P,Espanya,Bilbao,Home,1284.0,Grup B,Administratiu,True,...,1.0,,6.0,1361.04,1284.00,72.0,18.90,60,1,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Home,1074.0,Grup A,Data Analyst,True,...,0.0,,8.0,1159.92,537.00,68.0,17.06,64,2,
989,Chloé,Dubois,66354268T,Francia,Nice,Home,1954.0,Grup B,Data Analyst,False,...,0.0,Ciudad no encontrada,5.0,2051.70,977.00,NaN,NaN,27,1,Costo de vida inválido o cero (RPAL Infinito);
990,Adrià,Vila Díaz,57511543T,Espanya,Palma,Home,2010.0,Grup C,Analista,False,...,1.0,,5.5,2120.55,1005.00,73.0,29.05,34,1,
991,Sara,Serra,90585882A,Espanya,Tarragona,Dona,1073.0,Grup A,Administratiu,False,...,0.0,,5.0,1126.65,1073.00,67.0,16.82,24,0,


## Ejercicio 4: Exportación y síntesis de resultados

Crear el DataFrame resumido con agregaciones específicas
Redondear las métricas para que el reporte sea legible

In [ ]:

df_resumen = df.groupby('Grup Professional').agg(
    Total_Trabajadores=('Salario_Ajustado', 'count'), 
    Salario_Medio=('Salario_Ajustado', 'mean'),
    Edad_Media=('Edad', 'mean')
).reset_index()


df_resumen['Salario_Medio'] = df_resumen['Salario_Medio'].round(2)
df_resumen['Edad_Media'] = df_resumen['Edad_Media'].round(1)

Bucle para exportar los 4 archivos detallados por Grupo Profesional
Limpiamos el nombre del grupo para el nombre del archivo (ej: 'Grup A' -> 'GrupA')
Exportar detalle (index=False evita que se guarde la columna de índices numéricos)
 Exportar el archivo con el resumen agrupado

In [ ]:

for grupo, df_grupo in df.groupby('Grup Professional' \
''):
    
    nombre_archivo = f"datos_{str(grupo).replace(' ', '')}.csv"
    
    
    df_grupo.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    print(f"Archivo de detalle exportado: {nombre_archivo}")


df_resumen.to_csv("resumen_grupos_profesionales.csv", index=False, encoding='utf-8-sig')
print("Archivo de resumen exportado: resumen_grupos_profesionales.csv")

Archivo de detalle exportado: datos_GrupA.csv
Archivo de detalle exportado: datos_GrupB.csv
Archivo de detalle exportado: datos_GrupC.csv
Archivo de detalle exportado: datos_GrupD.csv
Archivo de resumen exportado: resumen_grupos_profesionales.csv


In [2407]:
df_grupo = pd.read_csv('E:/osley/Barecelona activa/curso especializacion analisis de datos/sprint 8/resumen_grupos_profesionales.csv', encoding='utf-8')

In [2408]:
display(df_grupo)

,Grup Professional,Total_Trabajadores,Salario_Medio,Edad_Media
0,Grup A,462,1187.45,43.5
1,Grup B,351,1714.96,44.2
2,Grup C,140,2395.30,45.8
3,Grup D,40,3284.08,41.6
